In [ ]:
# Libraries import

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings("ignore")

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pylab as plt

import seaborn as sns

In [ ]:
# Dataset import
dataset = pd.read_csv("Walmart_Store_sales.csv")

# PART 1

# Basic exploration

In [ ]:
# Getting number of rows and columns of dataset

dataset.shape

In [ ]:
# Printing first 5 rows of dataset

dataset.head()

In [ ]:
# Getting all statistics of dataset

dataset.describe(include='all')

In [ ]:
# Printing more info about dataset

dataset.info()

In [ ]:
#Display the % of missing values in each column of the dataset

dataset.isnull().sum()/dataset.shape[0]

# EDA

In [ ]:
fig1 = px.scatter(dataset, x = 'Temperature' , y = 'Weekly_Sales')
fig1.show()

In [ ]:
fig2 = px.scatter(dataset, x = 'Fuel_Price' , y = 'Weekly_Sales')

fig2.show()

In [ ]:
fig3 = px.scatter(dataset, x = 'CPI' , y = 'Weekly_Sales')
fig3.show()

In [ ]:
fig4 = px.scatter(dataset, x = 'Unemployment' , y = 'Weekly_Sales')
fig4.show()

In [ ]:
correlation_matrix = dataset.corr()

sns.heatmap(correlation_matrix, annot=True)

We can conclude from the EDA that it didn't helped us much to understand clear correlations between the target variable and the different features.

# Preprocessing with Pandas

Cleaning

In [ ]:
dataset.shape

In [ ]:
# Removing rows with missing values from target variable

dataset = dataset.dropna(subset = ["Weekly_Sales"])

In [ ]:
# Removing "Date" column from the dataset because we can't use it as it is for the model

dataset = dataset.drop("Date", axis = 1)

In [ ]:
dataset.shape

In [ ]:
# Removing outliers from the column "Temperature"

average_temperature = dataset["Temperature"].mean()
standard_deviation_temperature = dataset["Temperature"].std()
temp_to_keep_above = (dataset["Temperature"] > (average_temperature - 3 * standard_deviation_temperature))
temp_to_keep_under = (dataset["Temperature"] < (average_temperature + 3 * standard_deviation_temperature))
dataset = dataset.loc[temp_to_keep_above,:]
dataset = dataset.loc[temp_to_keep_under,:]

In [ ]:
# Removing outliers from the column "Fuel Price"

average_fuel_price = dataset["Fuel_Price"].mean()
standard_deviation_fuel_price = dataset["Fuel_Price"].std()
fuel_price_to_keep_above = (dataset["Fuel_Price"] > (average_fuel_price - 3 * standard_deviation_fuel_price))
fuel_price_to_keep_under = (dataset["Fuel_Price"] < (average_fuel_price + 3 * standard_deviation_fuel_price))
dataset = dataset.loc[fuel_price_to_keep_above,:]
dataset = dataset.loc[fuel_price_to_keep_under,:]

In [ ]:
# Removing outliers from the column "CPI"

average_cpi = dataset["CPI"].mean()
standard_deviation_cpi = dataset["CPI"].std()
cpi_to_keep_above = (dataset["CPI"] > (average_cpi - 3 * standard_deviation_cpi))
cpi_to_keep_under = (dataset["CPI"] < (average_cpi + 3 * standard_deviation_cpi))
dataset = dataset.loc[cpi_to_keep_above,:]
dataset = dataset.loc[cpi_to_keep_under,:]

In [ ]:
# Removing outliers from the column "Unemployment"

average_unemployment = dataset["Unemployment"].mean()
standard_deviation_unemployment = dataset["Unemployment"].std()
unemployment_to_keep_above = (dataset["Unemployment"] > (average_unemployment - 3 * standard_deviation_unemployment))
unemployment_to_keep_under = (dataset["Unemployment"] < (average_unemployment + 3 * standard_deviation_unemployment))
dataset = dataset.loc[unemployment_to_keep_above,:]
dataset = dataset.loc[unemployment_to_keep_under,:]

In [ ]:
dataset.shape

Preprocessing

In [ ]:
# Separating target variable Y from features X

print("Separating labels from features...")

feature_list = ['Store', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
target_variable =  'Weekly_Sales'

X = dataset.loc[:,feature_list]
Y = dataset.loc[:,target_variable]
print()
print("...Operation done")

# PART 2

# Preprocessing with Scikit-Learn

In [ ]:
# Splitting dataset into train and test set

print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, 
                                                    test_size=0.2, 
                                                    random_state=0)
print("...Done.")
print()

In [ ]:
# Standardizing numerical features at the same scale and encoding categorical features

numerical_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
categorical_features = ['Store', 'Holiday_Flag']

numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first')

feature_encoder = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

Training Pipeline

In [ ]:
# Training of the model on the train set

X_train = feature_encoder.fit_transform(X_train)

print("Train model...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")

In [ ]:
# Predictions on training set
print("Predictions on training set...")
Y_train_pred = regressor.predict(X_train)
print("...Done.")
print()
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))

Test Pipeline

In [ ]:
# Training of the model on the test set and predictions

X_test = feature_encoder.transform(X_test)

print("Predictions on test set...")
Y_test_pred = regressor.predict(X_test)
print("...Done.")
print()
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

We are overfitting a bit but in this case it's not significant at all because the model did almost perfect on the training set and the score of the model on the test set is still very high. 
We are going to try to reduce the overfitting by using a regularized linear regression model with Lasso and Ridge.

In [ ]:
# Performing a 10 fold cross validation to verify the standard deviation and check if we are really overfitting or not

scores = cross_val_score(regressor, X_train, Y_train, cv=10)
print('The cross-validated R2-score is : ', scores.mean())
print('The standard deviation is : ', scores.std())

The cross validation confirmed us that we are not overfitting at all because the standard deviation is near 0.

Coefficients

In [ ]:
# Checking the coefficients
print('coefficients are ', regressor.coef_)

In [ ]:
# Creation of a dataframe that takes the coefficients values in correlation with the good feature names
feature_names = feature_encoder.get_feature_names_out()
coeffs = abs(regressor.coef_)

df_coeffs = pd.DataFrame(
    coeffs,
    columns=["Coefficients"],
    index=feature_names,
)

In [ ]:
df_coeffs

In [ ]:
fig = px.bar(
    df_coeffs.sort_values(by="Coefficients", ascending= False),
    x ='Coefficients',
    y = df_coeffs.index,
    color = df_coeffs.index,
    title="Coefficients values per feature")

fig.update_layout(
    margin=dict(l=150, r=50, t=50, b=50),

)

fig.show()

The temperature, the fuel price, the CPI and the unemployment are the most important features with the biggest weight. The stores depending on which one have a relatively big weight too. The holiday flag relatively none even if the holiday flag related to special holiday week have a little weight.

# PART 3

# Regularized linear regression model

Ridge

In [ ]:
#Creation of the regressor to train a regularized model with Ridge

regressor_ridge = Ridge()
regressor_ridge.fit(X_train, Y_train)

print("Grid search...")

# Grid of values to be tested
params = {
    'alpha': [0, 0.1 , 0.5 , 1] # 0 corresponds to no regularization
}
gridsearch1 = GridSearchCV(regressor_ridge, param_grid = params, cv = 10) # cv : the number of folds to be used for CV
gridsearch1.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch1.best_params_)
print("Best R2 score : ", gridsearch1.best_score_)

In [ ]:
# Predictions on training set, the model have already been trained again on the training set with the grid search



print("Predictions on training set...")
Y_train_pred_grid1 = gridsearch1.predict(X_train)
print("...Done.")
print()
print("R2 score on train set after gridsearch : ", r2_score(Y_train, Y_train_pred_grid1))

In [ ]:
print("Predictions on test set...")

Y_test_pred_grid1 = gridsearch1.predict(X_test)
print("...Done.")
print()
print("R2 score on test set after gridsearch : ", r2_score(Y_test, Y_test_pred_grid1))

We have reduced a bit the overfitting (0,0406 with Ridge and 0,0451 with baseline) and improved the test score compared to the baseline model with no regularization. Let's try now with Lasso if we can do better.

Lasso

In [ ]:
#Creation of the regressor to train a regularized model with Lasso

regressor_lasso = Lasso(normalize=True)
regressor_lasso.fit(X_train, Y_train)

print("Grid search...")

# Grid of values to be tested
params = {
    'alpha': [0, 0.1 , 0.5 , 1] # 0 corresponds to no regularization
}
gridsearch2 = GridSearchCV(regressor_lasso, param_grid = params, cv = 10) # cv : the number of folds to be used for CV
gridsearch2.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch2.best_params_)
print("Best R2 score : ", gridsearch2.best_score_)

In [ ]:
# Predictions on training set, the model have already been trained again on the training set with the grid search



print("Predictions on training set...")
Y_train_pred_grid2 = gridsearch2.predict(X_train)
print("...Done.")
print()
print("R2 score on train set after gridsearch : ", r2_score(Y_train, Y_train_pred_grid2))

In [ ]:
print("Predictions on test set...")

Y_test_pred_grid2 = gridsearch2.predict(X_test)
print("...Done.")
print()
print("R2 score on test set after gridsearch : ", r2_score(Y_test, Y_test_pred_grid2))

This time with a Lasso regularization we have a tiny worst overfitting than with a Ridge regularization (0,0421 with Lasso and 0,0406 with Ridge) but it's still better than a baseline model with no regularization at all (0,0421 with Lasso and 0,0451 with baseline)